<a href="https://colab.research.google.com/github/Soljafree60/git-work/blob/main/05_PINN_Black_Scholes_LR_Schedule_Best_Checkpoint.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Improved PINN Black–Scholes Experiment
## Learning-Rate Schedule + Best-Model Checkpointing

This experiment keeps the previous PINN setup unchanged:

- same Black–Scholes parameters
- same 4 hidden layers with 64 `tanh` neurons each
- same 10,000 interior collocation points
- same 1,000 boundary points
- same 1,000 terminal points
- same normalised output
  $$
  u=\frac{V}{K}
  $$
- same loss weights
  $$
  \lambda_f=\lambda_B=\lambda_T=1
  $$

Only two controlled changes are introduced:

1. **Learning-rate decay**
   $$
   10^{-3}
   \rightarrow
   5\times10^{-4}
   \rightarrow
   10^{-4}
   $$

2. **Best-model checkpointing**

   The network weights are saved whenever the total PINN loss reaches a new minimum.  
   At the end of training, the best weights are restored before evaluation.

The analytical Black–Scholes solution is used only for final validation, not for training or model selection.


## 1. Imports and reproducibility


In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt
import time

from scipy.stats import norm

SEED = 42

np.random.seed(SEED)
tf.random.set_seed(SEED)

tf.keras.backend.set_floatx("float32")

print("TensorFlow version:", tf.__version__)
print("NumPy version:", np.__version__)


### Why I used this code

I used this cell to load the scientific-computing libraries required by the experiment. NumPy is used for numerical arrays and mathematical operations (Harris et al., 2020), pandas is used to organise numerical outputs into tables (McKinney, 2010), Matplotlib is used for figures (Hunter, 2007), SciPy provides established numerical routines such as the normal distribution and banded linear-system solvers (Virtanen et al., 2020), and TensorFlow supplies automatic differentiation and neural-network optimisation required by the PINN formulation (Raissi et al., 2019).



## 2. Black–Scholes parameters and PINN settings


In [ ]:
# ============================================================
# BLACK-SCHOLES PARAMETERS
# ============================================================

S0 = 100.0
K = 100.0
r = 0.05
sigma = 0.20
T = 1.0
S_max = 200.0

# ============================================================
# PINN SETTINGS
# ============================================================

N_f = 10_000
N_b = 1_000
N_T = 1_000

EPOCHS = 10_000

lambda_f = 1.0
lambda_B = 1.0
lambda_T = 1.0

# Learning-rate schedule
LR_1 = 1e-3
LR_2 = 5e-4
LR_3 = 1e-4

print("Baseline parameters")
print("-------------------")
print("S0 =", S0)
print("K =", K)
print("r =", r)
print("sigma =", sigma)
print("T =", T)
print("S_max =", S_max)

print("\nPINN settings")
print("-------------")
print("N_f =", N_f)
print("N_b =", N_b)
print("N_T =", N_T)
print("Epochs =", EPOCHS)
print("lambda_f =", lambda_f)
print("lambda_B =", lambda_B)
print("lambda_T =", lambda_T)

print("\nLearning-rate schedule")
print("----------------------")
print("Epochs 1-3000    :", LR_1)
print("Epochs 3001-7000 :", LR_2)
print("Epochs 7001-10000:", LR_3)


### Why I used these model parameters and hyperparameters

The Black–Scholes parameters define the financial problem being solved. The baseline values are held fixed unless the notebook is explicitly performing a sensitivity study. This produces a controlled reference case against which numerical changes can be measured (Black & Scholes, 1973; Hull, 2018).

The PINN hyperparameters define the computational budget and network training configuration. Keeping these settings explicit is important for reproducibility and makes it possible to distinguish changes in model accuracy caused by the market parameter from changes caused by the numerical procedure.


## 3. Analytical Black–Scholes solution for validation


In [ ]:
def black_scholes_call(S, K, r, sigma, T, t=0.0):
    """
    Analytical European call price.
    Handles S = 0 and t = T safely.
    """

    scalar_input = np.isscalar(S)
    S = np.asarray(S, dtype=float)

    tau = T - t

    if tau <= 0:
        result = np.maximum(S - K, 0.0)
        return float(result) if scalar_input else result

    result = np.zeros_like(S, dtype=float)

    positive = S > 0
    S_pos = S[positive]

    d1 = (
        np.log(S_pos / K)
        + (r + 0.5 * sigma**2) * tau
    ) / (sigma * np.sqrt(tau))

    d2 = d1 - sigma * np.sqrt(tau)

    result[positive] = (
        S_pos * norm.cdf(d1)
        - K * np.exp(-r * tau) * norm.cdf(d2)
    )

    return float(result) if scalar_input else result


analytical_price = black_scholes_call(
    S0, K, r, sigma, T, t=0.0
)

print(
    f"Analytical Black-Scholes price at S={S0:.0f}, t=0: "
    f"{analytical_price:.10f}"
)


### Why I used the analytical Black–Scholes function

The analytical European call solution is used as the benchmark because the Black–Scholes model has a closed-form solution under the assumptions used in this study (Black & Scholes, 1973; Hull, 2018). This allowed the numerical FDM and PINN solutions to be evaluated against a known reference rather than against one another.

The implementation handles $S=0$ and maturity separately to avoid undefined expressions such as $\log(0)$ or division by zero. These are numerical safeguards; they do not alter the Black–Scholes model.


## 4. Input scaling


In [ ]:
def scale_inputs(S, t):

    S_scaled = 2.0 * S / S_max - 1.0
    t_scaled = 2.0 * t / T - 1.0

    return S_scaled, t_scaled


def model_prediction(model, S, t):
    """
    Predict normalised option value u = V/K.
    """

    S_scaled, t_scaled = scale_inputs(S, t)

    inputs = tf.concat(
        [S_scaled, t_scaled],
        axis=1
    )

    return model(inputs)


### Why I scaled the inputs

The asset price and time variables have different numerical ranges. Scaling them to approximately $[-1,1]$ improves the conditioning of the neural-network optimisation and is compatible with the use of `tanh` activation functions. PINNs are sensitive to optimisation and gradient behaviour, so numerical scaling can materially improve training stability (Karniadakis et al., 2021; Wang et al., 2021).

The PDE derivatives are still taken with respect to the physical variables $S$ and $t$. TensorFlow applies the chain rule through the scaling transformation automatically.


## 5. PINN architecture


In [ ]:
class PINN(tf.keras.Model):

    def __init__(self):
        super().__init__()

        self.hidden1 = tf.keras.layers.Dense(
            64,
            activation="tanh"
        )

        self.hidden2 = tf.keras.layers.Dense(
            64,
            activation="tanh"
        )

        self.hidden3 = tf.keras.layers.Dense(
            64,
            activation="tanh"
        )

        self.hidden4 = tf.keras.layers.Dense(
            64,
            activation="tanh"
        )

        self.output_layer = tf.keras.layers.Dense(1)

    def call(self, inputs):

        x = self.hidden1(inputs)
        x = self.hidden2(x)
        x = self.hidden3(x)
        x = self.hidden4(x)

        return self.output_layer(x)


# Fresh model for this controlled experiment
model = PINN()

sample_S = tf.constant(
    [[100.0]],
    dtype=tf.float32
)

sample_t = tf.constant(
    [[0.5]],
    dtype=tf.float32
)

_ = model_prediction(
    model,
    sample_S,
    sample_t
)

model.summary()


### Why I used this PINN architecture

The network uses four hidden layers with 64 neurons per layer and `tanh` activations. Smooth activation functions are appropriate for PINNs because the PDE residual requires first and second derivatives of the network output. The architecture follows the general fully connected neural-network structure commonly used in PINN studies (Raissi et al., 2019; Karniadakis et al., 2021).

The output layer contains one neuron because the required solution is a scalar option value $V(S,t)$, represented in the improved implementation through the normalised variable $u=V/K$.


## 6. Generate collocation, boundary and terminal points


In [ ]:
# ============================================================
# INTERIOR COLLOCATION POINTS
# ============================================================

S_f = tf.random.uniform(
    shape=(N_f, 1),
    minval=0.0,
    maxval=S_max,
    dtype=tf.float32,
    seed=SEED
)

t_f = tf.random.uniform(
    shape=(N_f, 1),
    minval=0.0,
    maxval=T,
    dtype=tf.float32,
    seed=SEED + 1
)

# ============================================================
# BOUNDARY POINTS
# ============================================================

t_b = tf.random.uniform(
    shape=(N_b, 1),
    minval=0.0,
    maxval=T,
    dtype=tf.float32,
    seed=SEED + 2
)

S_left = tf.zeros(
    shape=(N_b, 1),
    dtype=tf.float32
)

S_right = tf.ones(
    shape=(N_b, 1),
    dtype=tf.float32
) * S_max

V_left = tf.zeros(
    shape=(N_b, 1),
    dtype=tf.float32
)

V_right = (
    S_max
    - K * tf.exp(
        -r * (T - t_b)
    )
)

# ============================================================
# TERMINAL POINTS
# ============================================================

S_T = tf.random.uniform(
    shape=(N_T, 1),
    minval=0.0,
    maxval=S_max,
    dtype=tf.float32,
    seed=SEED + 3
)

t_T = tf.ones(
    shape=(N_T, 1),
    dtype=tf.float32
) * T

V_T = tf.maximum(
    S_T - K,
    0.0
)

print("Interior points:", S_f.shape, t_f.shape)
print("Boundary points:", S_left.shape, S_right.shape)
print("Terminal points:", S_T.shape, t_T.shape)


### Why I generated collocation, boundary and terminal points

PINNs do not require a conventional labelled training dataset for the interior of the domain. Instead, collocation points are sampled in the $(S,t)$ domain and the governing PDE is enforced through its residual (Raissi et al., 2019).

Separate boundary and terminal samples are required because the Black–Scholes PDE alone does not uniquely determine the European call solution. The boundary conditions and maturity payoff supply the additional constraints needed to identify the correct solution.


## 7. Normalised targets


In [ ]:
U_left = V_left / K
U_right = V_right / K
U_T = V_T / K

print(
    "Normalised lower boundary range:",
    float(tf.reduce_min(U_left).numpy()),
    "to",
    float(tf.reduce_max(U_left).numpy())
)

print(
    "Normalised upper boundary range:",
    float(tf.reduce_min(U_right).numpy()),
    "to",
    float(tf.reduce_max(U_right).numpy())
)

print(
    "Normalised terminal range:",
    float(tf.reduce_min(U_T).numpy()),
    "to",
    float(tf.reduce_max(U_T).numpy())
)


### Why I normalised the option values

The PINN predicts

$$
u=\frac{V}{K}
$$

rather than the raw option value $V$. This was introduced after the preliminary experiment showed that the boundary and terminal losses were numerically much larger than the PDE loss. Normalising the targets reduces the scale imbalance between loss components and improves optimisation stability. Loss imbalance and gradient pathologies are recognised issues in PINN training (Wang et al., 2021).

Because $V=Ku$ and $K$ is constant within each experiment, the Black–Scholes PDE retains the same mathematical form after division by $K$.


## 8. PDE residual and automatic differentiation


In [ ]:
def pde_residual(model, S, t):

    with tf.GradientTape(
        persistent=True
    ) as tape2:

        tape2.watch(S)
        tape2.watch(t)

        with tf.GradientTape(
            persistent=True
        ) as tape1:

            tape1.watch(S)
            tape1.watch(t)

            u = model_prediction(
                model,
                S,
                t
            )

        u_S = tape1.gradient(
            u,
            S
        )

        u_t = tape1.gradient(
            u,
            t
        )

    u_SS = tape2.gradient(
        u_S,
        S
    )

    del tape1
    del tape2

    residual = (
        u_t
        + 0.5 * sigma**2 * S**2 * u_SS
        + r * S * u_S
        - r * u
    )

    return (
        u,
        u_t,
        u_S,
        u_SS,
        residual
    )


u_test, u_t_test, u_S_test, u_SS_test, R_test = pde_residual(
    model,
    S_f[:5],
    t_f[:5]
)

print("u shape:    ", u_test.shape)
print("u_t shape:  ", u_t_test.shape)
print("u_S shape:  ", u_S_test.shape)
print("u_SS shape: ", u_SS_test.shape)
print("R shape:    ", R_test.shape)

print("\nSample residuals before training:")
print(R_test.numpy())


### Why I used automatic differentiation for the PDE residual

The PINN solution is constrained by the Black–Scholes PDE. TensorFlow automatic differentiation is therefore used to obtain

$$
u_t,\qquad u_S,\qquad u_{SS}.
$$

These derivatives are substituted directly into the differential equation to form the residual. Minimising the residual at collocation points is the central physics-informed mechanism that allows the network to learn the PDE solution without labelled interior solution values (Raissi et al., 2019; Karniadakis et al., 2021).

Nested gradient tapes are required because $u_{SS}$ is a second derivative.


## 9. Loss functions


In [ ]:
def pde_loss(model, S_f, t_f):

    _, _, _, _, residual = pde_residual(
        model,
        S_f,
        t_f
    )

    return tf.reduce_mean(
        tf.square(residual)
    )


def boundary_loss(
    model,
    S_left,
    S_right,
    t_b,
    U_left,
    U_right
):

    U_left_pred = model_prediction(
        model,
        S_left,
        t_b
    )

    U_right_pred = model_prediction(
        model,
        S_right,
        t_b
    )

    U_pred = tf.concat(
        [U_left_pred, U_right_pred],
        axis=0
    )

    U_true = tf.concat(
        [U_left, U_right],
        axis=0
    )

    return tf.reduce_mean(
        tf.square(
            U_pred - U_true
        )
    )


def terminal_loss(
    model,
    S_T,
    t_T,
    U_T
):

    U_T_pred = model_prediction(
        model,
        S_T,
        t_T
    )

    return tf.reduce_mean(
        tf.square(
            U_T_pred - U_T
        )
    )


def total_loss(model):

    loss_pde = pde_loss(
        model,
        S_f,
        t_f
    )

    loss_boundary = boundary_loss(
        model,
        S_left,
        S_right,
        t_b,
        U_left,
        U_right
    )

    loss_terminal = terminal_loss(
        model,
        S_T,
        t_T,
        U_T
    )

    loss_total = (
        lambda_f * loss_pde
        + lambda_B * loss_boundary
        + lambda_T * loss_terminal
    )

    return (
        loss_total,
        loss_pde,
        loss_boundary,
        loss_terminal
    )


### Why I used three PINN loss components

The total PINN objective combines:

$$
L_{\mathrm{PDE}},\qquad
L_{\mathrm{boundary}},\qquad
L_{\mathrm{terminal}}.
$$

The PDE loss enforces the governing Black–Scholes equation inside the domain, while the boundary and terminal losses enforce the financial constraints that define the European call problem. Composite physics-informed losses of this type are standard in PINN formulations (Raissi et al., 2019).

Equal weights are used as the baseline so that the study begins from a transparent weighting scheme. The earlier normalisation step was used to reduce scale imbalance before introducing more complicated adaptive weighting.


## 10. Initial loss diagnostic


In [ ]:
(
    initial_total,
    initial_pde,
    initial_boundary,
    initial_terminal
) = total_loss(model)

print("Initial Total Loss:   ", float(initial_total.numpy()))
print("Initial PDE Loss:     ", float(initial_pde.numpy()))
print("Initial Boundary Loss:", float(initial_boundary.numpy()))
print("Initial Terminal Loss:", float(initial_terminal.numpy()))


## 11. Adam optimiser


In [ ]:
optimizer = tf.keras.optimizers.Adam(
    learning_rate=LR_1
)


@tf.function(reduce_retracing=True)
def train_step():

    with tf.GradientTape() as tape:

        (
            loss_total,
            loss_pde,
            loss_boundary,
            loss_terminal
        ) = total_loss(model)

    gradients = tape.gradient(
        loss_total,
        model.trainable_variables
    )

    gradient_variable_pairs = [
        (g, v)
        for g, v in zip(
            gradients,
            model.trainable_variables
        )
        if g is not None
    ]

    optimizer.apply_gradients(
        gradient_variable_pairs
    )

    return (
        loss_total,
        loss_pde,
        loss_boundary,
        loss_terminal
    )


### Why I used Adam and gradient-based training

Adam is used to update the trainable neural-network parameters by minimising the total PINN loss. Gradient-based optimisation is necessary because the PINN parameters are learned rather than obtained from a direct linear-system solve (Raissi et al., 2019).

The outer `GradientTape` differentiates the total loss with respect to the network weights and biases. This is different from the derivative tapes used inside the PDE residual, which differentiate the network output with respect to $S$ and $t$.


## 12. Train for 10,000 epochs with learning-rate decay and best-model tracking

Schedule:

$$
\eta=
\begin{cases}
10^{-3}, & 1\leq epoch\leq3000\\
5\times10^{-4}, & 3001\leq epoch\leq7000\\
10^{-4}, & 7001\leq epoch\leq10000
\end{cases}
$$

The **best model** is selected using the smallest total PINN loss, not the analytical Black–Scholes error.  
This avoids using the analytical benchmark to guide training.


In [ ]:
loss_history = []
pde_history = []
boundary_history = []
terminal_history = []
learning_rate_history = []

best_loss = np.inf
best_epoch = 0
best_weights = None

start_time = time.perf_counter()

for epoch in range(
    1,
    EPOCHS + 1
):

    # --------------------------------------------------------
    # LEARNING-RATE SCHEDULE
    # --------------------------------------------------------

    if epoch == 3001:
        optimizer.learning_rate.assign(
            LR_2
        )

        print(
            "\nLearning rate changed to",
            LR_2,
            "at epoch",
            epoch
        )

    if epoch == 7001:
        optimizer.learning_rate.assign(
            LR_3
        )

        print(
            "\nLearning rate changed to",
            LR_3,
            "at epoch",
            epoch
        )

    # --------------------------------------------------------
    # TRAINING STEP
    # --------------------------------------------------------

    (
        loss_total,
        loss_pde,
        loss_boundary,
        loss_terminal
    ) = train_step()

    current_total = float(
        loss_total.numpy()
    )

    # --------------------------------------------------------
    # STORE LOSS HISTORY
    # --------------------------------------------------------

    loss_history.append(
        current_total
    )

    pde_history.append(
        float(loss_pde.numpy())
    )

    boundary_history.append(
        float(loss_boundary.numpy())
    )

    terminal_history.append(
        float(loss_terminal.numpy())
    )

    learning_rate_history.append(
        float(
            optimizer.learning_rate.numpy()
        )
    )

    # --------------------------------------------------------
    # BEST MODEL CHECKPOINT
    # --------------------------------------------------------

    if current_total < best_loss:

        best_loss = current_total
        best_epoch = epoch

        best_weights = model.get_weights()

    # --------------------------------------------------------
    # PROGRESS OUTPUT
    # --------------------------------------------------------

    if epoch == 1 or epoch % 1000 == 0:

        print(
            f"Epoch {epoch:5d} | "
            f"LR: {optimizer.learning_rate.numpy():.1e} | "
            f"Total: {loss_total.numpy():.6e} | "
            f"PDE: {loss_pde.numpy():.6e} | "
            f"Boundary: {loss_boundary.numpy():.6e} | "
            f"Terminal: {loss_terminal.numpy():.6e}"
        )

end_time = time.perf_counter()

training_time = (
    end_time - start_time
)

print("\nTraining completed.")
print(
    f"Total training time: "
    f"{training_time:.6f} seconds"
)

print(
    f"Best epoch: {best_epoch}"
)

print(
    f"Best total loss: {best_loss:.10e}"
)


### Why staged learning-rate decay and best-model checkpointing are used

The earlier fixed-learning-rate run showed repeated loss spikes late in training. The learning rate is therefore reduced in stages so that the optimiser can make larger updates early in training and smaller, more stable updates once it approaches a low-loss region. PINN optimisation difficulties and gradient pathologies are well documented (Wang et al., 2021; Karniadakis et al., 2021).

Best-model checkpointing stores the network parameters whenever the total PINN loss reaches a new minimum. The final evaluation then uses the lowest-loss model rather than automatically assuming that the last epoch is the best one. Importantly, the analytical Black–Scholes error is not used for checkpoint selection, so the exact benchmark does not leak into the training decision.


## 13. Restore the best model weights


In [ ]:
if best_weights is None:
    raise RuntimeError(
        "No best-model weights were recorded."
    )

model.set_weights(
    best_weights
)

print(
    "Best model restored from epoch:",
    best_epoch
)

(
    restored_total,
    restored_pde,
    restored_boundary,
    restored_terminal
) = total_loss(model)

print(
    "Restored Total Loss:   ",
    float(restored_total.numpy())
)

print(
    "Restored PDE Loss:     ",
    float(restored_pde.numpy())
)

print(
    "Restored Boundary Loss:",
    float(restored_boundary.numpy())
)

print(
    "Restored Terminal Loss:",
    float(restored_terminal.numpy())
)


## 14. Training-loss graph


In [ ]:
plt.figure(
    figsize=(10, 6)
)

plt.semilogy(
    loss_history,
    label="Total Loss"
)

plt.semilogy(
    pde_history,
    label="PDE Loss"
)

plt.semilogy(
    boundary_history,
    label="Boundary Loss"
)

plt.semilogy(
    terminal_history,
    label="Terminal Loss"
)

plt.axvline(
    3000,
    linestyle="--",
    label="LR change: 5e-4"
)

plt.axvline(
    7000,
    linestyle="--",
    label="LR change: 1e-4"
)

plt.xlabel(
    "Epoch"
)

plt.ylabel(
    "Loss"
)

plt.title(
    "PINN Training Loss with Learning-Rate Decay"
)

plt.legend()
plt.grid(True)

plt.show()


### Why I used a logarithmic scale for the loss graph

PINN losses often change by several orders of magnitude during training. A logarithmic vertical axis makes both the early large losses and the later small losses visible on the same figure. The graph is used to assess convergence and identify instability or spikes in the optimisation process. Matplotlib is used for the visualisation (Hunter, 2007).


## 15. Learning-rate schedule graph


In [ ]:
plt.figure(
    figsize=(10, 5)
)

plt.plot(
    learning_rate_history
)

plt.xlabel(
    "Epoch"
)

plt.ylabel(
    "Learning Rate"
)

plt.title(
    "PINN Learning-Rate Schedule"
)

plt.grid(True)

plt.show()


### Why I included this figure

The figure provides a visual comparison that complements the numerical error table. Curves that appear close on an option-price plot may still have materially different numerical errors, so the graphical results are interpreted together with MSE, RMSE, $L_2$ and maximum-error measures.




## 16. Evaluate best PINN at S = 100, t = 0


In [ ]:
S_point = tf.constant(
    [[S0]],
    dtype=tf.float32
)

t_point = tf.constant(
    [[0.0]],
    dtype=tf.float32
)

u_point = model_prediction(
    model,
    S_point,
    t_point
)

V_pinn_point = (
    K * u_point
)

pinn_price = float(
    V_pinn_point.numpy()[0, 0]
)

absolute_point_error = abs(
    pinn_price
    - analytical_price
)

relative_point_error_pct = (
    absolute_point_error
    / analytical_price
    * 100.0
)

print(
    f"Best PINN Call Price: "
    f"{pinn_price:.10f}"
)

print(
    f"Analytical Call Price: "
    f"{analytical_price:.10f}"
)

print(
    f"Absolute Error: "
    f"{absolute_point_error:.10f}"
)

print(
    f"Relative Error: "
    f"{relative_point_error_pct:.6f}%"
)


## 17. Evaluate best PINN across asset-price domain at t = 0


In [ ]:
S_eval_np = np.linspace(
    0.0,
    S_max,
    401
).reshape(-1, 1)

t_eval_np = np.zeros_like(
    S_eval_np
)

S_eval_tf = tf.constant(
    S_eval_np,
    dtype=tf.float32
)

t_eval_tf = tf.constant(
    t_eval_np,
    dtype=tf.float32
)

u_pinn = model_prediction(
    model,
    S_eval_tf,
    t_eval_tf
).numpy().reshape(-1)

V_pinn = (
    K * u_pinn
)

V_exact = black_scholes_call(
    S_eval_np.reshape(-1),
    K,
    r,
    sigma,
    T,
    t=0.0
)

error = (
    V_pinn
    - V_exact
)

mse = np.mean(
    error**2
)

rmse = np.sqrt(
    mse
)

l2_error = np.linalg.norm(
    error,
    ord=2
)

max_abs_error = np.max(
    np.abs(error)
)

print(
    "Best PINN evaluation metrics at t = 0"
)

print(
    "------------------------------------"
)

print(
    f"MSE:                {mse:.10e}"
)

print(
    f"RMSE:               {rmse:.10e}"
)

print(
    f"L2 Error:           {l2_error:.10e}"
)

print(
    f"Maximum Abs Error:  {max_abs_error:.10e}"
)

print(
    f"Training Runtime:    {training_time:.6f} seconds"
)


## 18. Analytical Black–Scholes vs best PINN


In [ ]:
plt.figure(
    figsize=(10, 6)
)

plt.plot(
    S_eval_np.reshape(-1),
    V_exact,
    label="Analytical Black-Scholes"
)

plt.plot(
    S_eval_np.reshape(-1),
    V_pinn,
    "--",
    label="Improved PINN"
)

plt.xlabel(
    "Asset Price (S)"
)

plt.ylabel(
    "European Call Option Price"
)

plt.title(
    "Analytical Black-Scholes vs Improved PINN"
)

plt.legend()
plt.grid(True)

plt.show()


### Why I included this figure

The figure provides a visual comparison that complements the numerical error table. Curves that appear close on an option-price plot may still have materially different numerical errors, so the graphical results are interpreted together with MSE, RMSE, $L_2$ and maximum-error measures.



## 19. Best PINN error curve


In [ ]:
plt.figure(
    figsize=(10, 6)
)

plt.plot(
    S_eval_np.reshape(-1),
    error
)

plt.axhline(
    0.0,
    linestyle="--"
)

plt.xlabel(
    "Asset Price (S)"
)

plt.ylabel(
    "PINN Error"
)

plt.title(
    "Improved PINN Error Relative to Analytical Black-Scholes"
)

plt.grid(True)

plt.show()


### Why I included this figure

The figure provides a visual comparison that complements the numerical error table. Curves that appear close on an option-price plot may still have materially different numerical errors, so the graphical results are interpreted together with MSE, RMSE, $L_2$ and maximum-error measures.



## 20. Full best-PINN solution


In [ ]:
S_surface = np.linspace(
    0.0,
    S_max,
    200
)

t_surface = np.linspace(
    0.0,
    T,
    100
)

S_mesh, t_mesh = np.meshgrid(
    S_surface,
    t_surface
)

S_flat = tf.constant(
    S_mesh.reshape(-1, 1),
    dtype=tf.float32
)

t_flat = tf.constant(
    t_mesh.reshape(-1, 1),
    dtype=tf.float32
)

u_surface = model_prediction(
    model,
    S_flat,
    t_flat
).numpy().reshape(
    t_mesh.shape
)

V_surface = (
    K * u_surface
)

plt.figure(
    figsize=(10, 6)
)

contour = plt.contourf(
    S_mesh,
    t_mesh,
    V_surface,
    levels=50
)

plt.colorbar(
    contour,
    label="Call Option Price"
)

plt.xlabel(
    "Asset Price (S)"
)

plt.ylabel(
    "Time (t)"
)

plt.title(
    "Improved PINN Black-Scholes Solution"
)

plt.show()


### Why I included this figure

The figure provides a visual comparison that complements the numerical error table. Curves that appear close on an option-price plot may still have materially different numerical errors, so the graphical results are interpreted together with MSE, RMSE, $L_2$ and maximum-error measures.


## 21. Full-domain absolute error


In [ ]:
V_exact_surface = np.zeros_like(
    S_mesh,
    dtype=float
)

for j, t_value in enumerate(
    t_surface
):

    V_exact_surface[j, :] = black_scholes_call(
        S_surface,
        K,
        r,
        sigma,
        T,
        t=float(t_value)
    )

absolute_error_surface = np.abs(
    V_surface
    - V_exact_surface
)

plt.figure(
    figsize=(10, 6)
)

contour = plt.contourf(
    S_mesh,
    t_mesh,
    absolute_error_surface,
    levels=50
)

plt.colorbar(
    contour,
    label="Absolute Error"
)

plt.xlabel(
    "Asset Price (S)"
)

plt.ylabel(
    "Time (t)"
)

plt.title(
    "Improved PINN Absolute Error Across the Black-Scholes Domain"
)

plt.show()


### Why I included this figure

The figure provides a visual comparison that complements the numerical error table. Curves that appear close on an option-price plot may still have materially different numerical errors, so the graphical results are interpreted together with MSE, RMSE, $L_2$ and maximum-error measures.



## 22. Summary table


In [ ]:
summary = pd.DataFrame({
    "Method": [
        "Improved PINN"
    ],
    "Epochs": [
        EPOCHS
    ],
    "Best Epoch": [
        best_epoch
    ],
    "Best Total Loss": [
        best_loss
    ],
    "Price at S=100": [
        pinn_price
    ],
    "Analytical Price": [
        analytical_price
    ],
    "Absolute Price Error": [
        absolute_point_error
    ],
    "Relative Price Error (%)": [
        relative_point_error_pct
    ],
    "MSE": [
        mse
    ],
    "RMSE": [
        rmse
    ],
    "L2 Error": [
        l2_error
    ],
    "Maximum Abs Error": [
        max_abs_error
    ],
    "Training Runtime (s)": [
        training_time
    ]
})

summary


## 23. What to compare with the previous 10,000-epoch PINN

Previous baseline 10,000-epoch PINN:

- Price at S=100: 9.2347011566
- Analytical price: 10.4505835722
- Absolute price error: 1.2158824156
- MSE: 1.6600546608
- RMSE: 1.2884310850
- L2 error: 25.800812370
- Maximum absolute error: 1.9940496127
- Training runtime: 1879.494985 seconds



## Results obtained from the improved PINN experiment

The improved experiment retained the same architecture and loss formulation but introduced staged learning-rate decay and best-model checkpointing.

| Measure | Result |
|---|---:|
| Best epoch | 10,000 |
| Best total loss | $1.811079\times10^{-5}$ |
| Restored PDE loss | $6.444697\times10^{-6}$ |
| Restored boundary loss | $1.592800\times10^{-7}$ |
| Restored terminal loss | $1.150586\times10^{-5}$ |
| Training time | 1779.100 s |
| PINN price at $S=100,t=0$ | 10.422435 |
| Analytical price | 10.450584 |
| Absolute pricing error | 0.028149 |
| Relative pricing error | 0.269351% |
| MSE | 0.001836 |
| RMSE | 0.042849 |
| $L_2$ error | 0.858043 |
| Maximum absolute error | 0.119248 |

The improvement over the fixed-learning-rate experiment supports the decision to retain the staged learning-rate schedule for the subsequent sensitivity analyses.




## References — CPUT Harvard style

Black, F. & Scholes, M. 1973. The pricing of options and corporate liabilities. *Journal of Political Economy*, 81(3):637-654. DOI: 10.1086/260062.

Crank, J. & Nicolson, P. 1947. A practical method for numerical evaluation of solutions of partial differential equations of the heat-conduction type. *Proceedings of the Cambridge Philosophical Society*, 43(1):50-67. DOI: 10.1017/S0305004100023197.

Duffy, D.J. 2006. *Finite difference methods in financial engineering: A partial differential equation approach*. Chichester: John Wiley & Sons. DOI: 10.1002/9781118673447.

Harris, C.R., Millman, K.J., van der Walt, S.J., Gommers, R., Virtanen, P., Cournapeau, D., Wieser, E., Taylor, J., Berg, S., Smith, N.J., Kern, R., Picus, M., Hoyer, S., van Kerkwijk, M.H., Brett, M., Haldane, A., del Río, J.F., Wiebe, M., Peterson, P., Gérard-Marchant, P., Sheppard, K., Reddy, T., Weckesser, W., Abbasi, H., Gohlke, C. & Oliphant, T.E. 2020. Array programming with NumPy. *Nature*, 585:357-362. DOI: 10.1038/s41586-020-2649-2.

Hull, J.C. 2018. *Options, futures, and other derivatives*. 10th ed. Harlow: Pearson.

Hunter, J.D. 2007. Matplotlib: A 2D graphics environment. *Computing in Science & Engineering*, 9(3):90-95. DOI: 10.1109/MCSE.2007.55.

Karniadakis, G.E., Kevrekidis, I.G., Lu, L., Perdikaris, P., Wang, S. & Yang, L. 2021. Physics-informed machine learning. *Nature Reviews Physics*, 3(6):422-440. DOI: 10.1038/s42254-021-00314-5.

McKinney, W. 2010. Data structures for statistical computing in Python. In: van der Walt, S. & Millman, J. eds. *Proceedings of the 9th Python in Science Conference*. Austin, TX: SciPy, 56-61. DOI: 10.25080/Majora-92bf1922-00a.

Raissi, M., Perdikaris, P. & Karniadakis, G.E. 2019. Physics-informed neural networks: A deep learning framework for solving forward and inverse problems involving nonlinear partial differential equations. *Journal of Computational Physics*, 378:686-707. DOI: 10.1016/j.jcp.2018.10.045.

Virtanen, P., Gommers, R., Oliphant, T.E., Haberland, M., Reddy, T., Cournapeau, D., Burovski, E., Peterson, P., Weckesser, W., Bright, J., van der Walt, S.J., Brett, M., Wilson, J., Millman, K.J., Mayorov, N., Nelson, A.R.J., Jones, E., Kern, R., Larson, E., Carey, C.J., Polat, İ., Feng, Y., Moore, E.W., VanderPlas, J., Laxalde, D., Perktold, J., Cimrman, R., Henriksen, I., Quintero, E.A., Harris, C.R., Archibald, A.M., Ribeiro, A.H., Pedregosa, F., van Mulbregt, P. & SciPy 1.0 Contributors. 2020. SciPy 1.0: Fundamental algorithms for scientific computing in Python. *Nature Methods*, 17:261-272. DOI: 10.1038/s41592-019-0686-2.

Wang, S., Teng, Y. & Perdikaris, P. 2021. Understanding and mitigating gradient flow pathologies in physics-informed neural networks. *SIAM Journal on Scientific Computing*, 43(5):A3055-A3081. DOI: 10.1137/20M1318043.

